# MCE Temporal Features: Maya Calendar Encoding

This notebook explores the MayaCalendarEncoder for temporal feature engineering,
demonstrating how Maya calendar cycles can enrich time series data.

In [ ]:
import numpy as np
from maya_encoding import MayaCalendarEncoder
from maya_encoding.core.calendar import (
    gregorian_to_jdn, jdn_to_tzolkin, jdn_to_haab,
    jdn_to_long_count, is_wayeb,
    TZOLKIN_DAY_NAMES, HAAB_MONTH_NAMES,
)

## 1. The Three Maya Calendar Systems

### Tzolk'in (Sacred Calendar)
- 260-day cycle = 13 numbers × 20 day names
- The two cycles (13 and 20) are coprime, creating 260 unique combinations

### Haab' (Solar Calendar)
- 365-day cycle = 18 months × 20 days + 5 "Wayeb'" days
- Each month has 20 days (0-19), plus 5 unlucky days

### Long Count
- Linear count from Maya epoch (August 11, 3114 BCE)
- Mixed radix: 20 (kin), 18 (uinal), 20 (tun), 20 (k'atun), 20 (b'ak'tun)

In [ ]:
# Explore notable dates
dates = [
    ("2012-12-21", "End of 13th b'ak'tun"),
    ("2024-01-01", "New Year 2024"),
    ("2024-03-20", "Spring Equinox 2024"),
    ("2024-06-21", "Summer Solstice 2024"),
    ("2024-12-21", "Winter Solstice 2024"),
    ("2000-01-01", "Y2K"),
]

print(f"{'Date':<14} {'Tzolkin':<15} {'Haab':<15} {'Long Count':<15} Note")
print("-" * 75)
for date_str, note in dates:
    jdn = gregorian_to_jdn(date_str)
    tz = jdn_to_tzolkin(jdn)
    hb = jdn_to_haab(jdn)
    lc = jdn_to_long_count(jdn)
    tz_str = f"{tz[0]} {TZOLKIN_DAY_NAMES[tz[1]]}"
    hb_str = f"day {hb[1]} mo {hb[0]}"
    lc_str = '.'.join(str(x) for x in lc)
    print(f"{date_str:<14} {tz_str:<15} {hb_str:<15} {lc_str:<15} {note}")

Date           Tzolkin         Haab            Long Count      Note
---------------------------------------------------------------------------
2012-12-21     4 Ajaw          day 3 mo 13     13.0.0.0.0      End of 13th b'ak'tun
2024-01-01     2 Lamat         day 16 mo 13    13.0.11.3.8     New Year 2024
2024-03-20     3 Manik'        day 15 mo 17    13.0.11.7.7     Spring Equinox 2024
2024-06-21     5 Ajaw          day 3 mo 4      13.0.11.12.0    Summer Solstice 2024
2024-12-21     6 Ak'bal        day 6 mo 13     13.0.12.3.3     Winter Solstice 2024
2000-01-01     11 Ik'          day 10 mo 13    12.19.6.15.2    Y2K


In [ ]:
# Show all 20 Tzolk'in day names
print("The 20 Tzolk'in day names:")
for i, name in enumerate(TZOLKIN_DAY_NAMES):
    print(f"  {i:>2}: {name}")

print(f"\nThe 19 Haab' months (18 regular + Wayeb'):")
for i, name in enumerate(HAAB_MONTH_NAMES):
    suffix = " (5 unlucky days)" if name == "Wayeb'" else ""
    print(f"  {i:>2}: {name}{suffix}")

The 20 Tzolk'in day names:
   0: Imix
   1: Ik'
   2: Ak'bal
   3: K'an
   4: Chikchan
   5: Kimi
   6: Manik'
   7: Lamat
   8: Muluk
   9: Ok
  10: Chuwen
  11: Eb
  12: Ben
  13: Ix
  14: Men
  15: Kib
  16: Kaban
  17: Etz'nab
  18: Kawak
  19: Ajaw

The 19 Haab' months (18 regular + Wayeb'):
   0: Pop
   1: Wo
   2: Sip
   3: Sotz'
   4: Sek
   5: Xul
   6: Yaxk'in
   7: Mol
   8: Ch'en
   9: Yax
  10: Sak
  11: Keh
  12: Mak
  13: K'ank'in
  14: Muwan
  15: Pax
  16: K'ayab
  17: Kumk'u
  18: Wayeb' (5 unlucky days)


## 2. Encoding Configurations

The MayaCalendarEncoder supports multiple encoding strategies.

In [ ]:
dates = np.array(["2024-01-01", "2024-06-15", "2024-12-21"])

# Tzolk'in only, separate components
enc = MayaCalendarEncoder(
    components=["tzolkin"],
    tzolkin_encoding="separate",
    cyclical=False,
)
result = enc.fit_transform(dates)
print("Tzolk'in (separate, no cyclical):")
print(f"  Features: {list(enc.get_feature_names_out())}")
print(f"  Shape: {result.shape}")
print(f"  Values:\n{result}")

Tzolk'in (separate, no cyclical):
  Features: ['tzolkin_number', 'tzolkin_day_name']
  Shape: (3, 2)
  Values:
[[0.08333333 0.36842105]
 [0.91666667 0.68421053]
 [0.41666667 0.10526316]]


In [ ]:
# Cyclical encoding adds sin/cos pairs
enc_cyc = MayaCalendarEncoder(
    components=["tzolkin"],
    tzolkin_encoding="separate",
    cyclical=True,
)
result_cyc = enc_cyc.fit_transform(dates)
print("Tzolk'in (separate, with cyclical sin/cos):")
print(f"  Features: {list(enc_cyc.get_feature_names_out())}")
print(f"  Shape: {result_cyc.shape}")
print(f"  Range: [{result_cyc.min():.4f}, {result_cyc.max():.4f}]")

Tzolk'in (separate, with cyclical sin/cos):
  Features: ['tzolkin_number', 'tzolkin_day_name', 'tzolkin_number_sin', 'tzolkin_number_cos', 'tzolkin_day_name_sin', 'tzolkin_day_name_cos']
  Shape: (3, 6)
  Range: [-0.8230, 0.9167]


In [ ]:
# Full encoding with all components
enc_full = MayaCalendarEncoder(
    components=["tzolkin", "haab", "long_count"],
    cyclical=True,
    wayeb_flag=True,
    long_count_levels=3,
)
result_full = enc_full.fit_transform(dates)
print(f"Full encoding: {result_full.shape[1]} features")
for name in enc_full.get_feature_names_out():
    print(f"  {name}")

Full encoding: 24 features
  tzolkin_number
  tzolkin_day_name
  tzolkin_number_sin
  tzolkin_number_cos
  tzolkin_day_name_sin
  tzolkin_day_name_cos
  haab_month
  haab_day
  haab_day_bars
  haab_day_dots
  haab_month_sin
  haab_month_cos
  haab_day_sin
  haab_day_cos
  is_wayeb
  long_count_kin
  long_count_kin_sin
  long_count_kin_cos
  long_count_uinal
  long_count_uinal_sin
  long_count_uinal_cos
  long_count_tun
  long_count_tun_sin
  long_count_tun_cos


## 3. Cycle Visualization

Let's visualize how the different calendar cycles create unique patterns.

In [ ]:
from datetime import datetime, timedelta

# Generate a year of dates
start = datetime(2024, 1, 1)
year_dates = [(start + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(365)]
year_array = np.array(year_dates)

# Get Tzolk'in positions
enc_tz = MayaCalendarEncoder(
    components=["tzolkin"],
    tzolkin_encoding="combined",
    cyclical=False, normalize=False
)
tz_vals = enc_tz.fit_transform(year_array)

print(f"Tzolk'in positions over 365 days:")
print(f"  Min: {tz_vals.min()}, Max: {tz_vals.max()}")
print(f"  Unique values: {len(np.unique(tz_vals))}")
print(f"  (260 unique in a 260-day cycle)")

# Count Wayeb' days
wayeb_count = sum(
    1 for d in year_dates
    if is_wayeb(gregorian_to_jdn(d))
)
print(f"\nWayeb' days in 2024: {wayeb_count} (expected ~5)")

Tzolk'in positions over 365 days:
  Min: 0.0, Max: 259.0
  Unique values: 260
  (260 unique in a 260-day cycle)

Wayeb' days in 2024: 5 (expected ~5)


## 4. Practical Usage in ML Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Synthetic time series with a 13-day cycle (Tzolk'in number)
np.random.seed(42)
n_days = 500
base = datetime(2020, 1, 1)
dates_train = np.array(
    [(base + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(n_days)]
)

# Target has a 13-day periodic component
jdns = np.array([gregorian_to_jdn(d) for d in dates_train])
y = np.sin(2 * np.pi * jdns / 13) * 5 + np.random.normal(0, 1, n_days)

# MCE should capture this 13-day cycle naturally
pipe = Pipeline([
    ("mce", MayaCalendarEncoder(
        components=["tzolkin"], cyclical=True
    )),
    ("rf", RandomForestRegressor(n_estimators=50, random_state=42)),
])

# Train/test split
split = int(n_days * 0.8)
pipe.fit(dates_train[:split], y[:split])
score = pipe.score(dates_train[split:], y[split:])
print(f"R² on test set (13-day cycle): {score:.4f}")
print("MCE naturally captures the Tzolk'in 13-number cycle!")

R² on test set (13-day cycle): 0.8920
MCE naturally captures the Tzolk'in 13-number cycle!
